# import

In [2]:
import pandas as pd
import os
import datetime
from datetime import datetime
from datetime import timedelta

import numpy as np
import glob
import json
import random
import matplotlib.pyplot as plt
import seaborn as sns
from multiprocessing import Pool
import time
import gc
import math


from random import sample
# import plotnine
# from plotnine import *
# import Image
from PIL import Image

from pandarallel import pandarallel


#import yqg_common as yqg
import csv
# import setproctitle
# from impala.dbapi import connect
import sys


from sklearn.model_selection import train_test_split
import xgboost as xgb
import lightgbm as lgb
from lightgbm import log_evaluation, early_stopping

from collections import Counter
from collections import defaultdict
from sklearn import metrics
# from sklearn import metrics.roc_auc_score
from collections import defaultdict

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn import preprocessing
from sklearn.metrics import roc_curve, auc

pd.set_option('display.max_rows', 10000)
pd.set_option('display.max_columns', 10000)
pd.set_option('display.width', 10000)
pd.set_option("max_colwidth", 10000)


# parallel
# from pandarallel import pandarallel
# pandarallel.initialize()

import warnings 
warnings.filterwarnings('ignore')

def auc_and_ks(results, model_score_column, label):
    auc = metrics.roc_auc_score(results[label], results[model_score_column])
    
    return auc

# import sys
sys.path.append('/data1/mex_reloan_data/code_')  # 将package所在目录加入执行目录列表
import modelEvaluation
import psiCalculation

import matplotlib.font_manager as fm
 
myfont = fm.FontProperties(fname='/data1/mex_reloan_data/SimHei.ttf',size=15) 

## Refactored pipeline (`model_pipeline`)

Train/load/split/predict/tune live in the **`model_pipeline`** package at the repo root. Install with `pip install -e ".[dev]"` from the project root, or add the repo root to `sys.path`, then see **`specs/model_pipeline/quickstart.md`** for a runnable minimal example.

In [ ]:
# Example imports (requires: pip install -e ".[dev]" from repo root, or sys.path to repo root):
from model_pipeline import ModelType, load_dataset, split_data, train, predict_scores

# train model code 

In [22]:
def lgb_model_train_and_save(train_data: pd.core.frame.DataFrame, # train数据
                              val_data: pd.core.frame.DataFrame, # validation数据 可以传None
                             label: str, # 训练标签 0/1
                             features: list, # 训练特征集
                             model_and_feats_file: str, # 模型和特征重要度文件存储路径
                             model_file_name: str, # 模型名称
                             feature_importance_file_name: str, # 特征重要度文件名称
                             is_early_stoppping: bool, # 是否使用early-stopping
                             num_boost_round: int, # 最大训练轮数
                             params: dict # 训练参数
                             ):
    if val_data is not None: # 如果提供val_data,则直接使用做validation集合，且默认走early-stopping
        # 
        lgtrain = lgb.Dataset(train_data[features], label = train_data[label], feature_name = features)
        lgval = lgb.Dataset(val_data[features], label = val_data[label], feature_name = features)

        callbacks = [log_evaluation(period=600), early_stopping(stopping_rounds=100)]
        valid_sets = [lgtrain, lgval]
        
    else: # 如果未提供val_data,且训练选择early-stopping模式，则按照默认方式划分train和val样本
        
        if is_early_stoppping:
            
            train_X, val_X, train_y, val_y = train_test_split(train_data[features], 
                                      data[label], 
                                                      test_size = 0.25, 
                                                      random_state=30, 
                                                      shuffle=True)
            # 参数调整 1
            lgtrain = lgb.Dataset(train_X, label = train_y, feature_name = features)
            lgval = lgb.Dataset(val_X, label = val_y, feature_name = features)

            del train_X, train_y, val_X, val_y
            gc.collect()

            callbacks = [log_evaluation(period=600), early_stopping(stopping_rounds=100)]
            valid_sets = [lgtrain, lgval]
            
        else: # 如果未提供val_data,且训练未选择early-stopping模式，则只使用train_data训练
        
            lgtrain = lgb.Dataset(train_data[features], label = train_data[label], feature_name = features)

            del train_data
            gc.collect()

            callbacks = [log_evaluation(period=600)]
            valid_sets = [lgtrain]
        
    
    model = lgb.train(params, 
                      lgtrain, 
                      num_boost_round = num_boost_round, 
                      valid_sets = valid_sets, 
                      callbacks = callbacks)
    
    model.save_model(f'{model_and_feats_file}/{model_file_name}.txt')
    
    print('save model done !!!!!!!!!!!')
    
    importance = model.feature_importance(importance_type='gain')

    '''
    importance_type (str, optional (default="split")) – 
    What type of feature importance should be dumped. 
    If “split”, result contains numbers of times the feature is used in a model. 
    If “gain”, result contains total gains of splits which use the feature.
    '''

    feature_name = model.feature_name()
    # for (feature_name,importance) in zip(feature_name,importance):
    #     print (feature_name,importance) 

    feature_importance = pd.DataFrame({
      'feature_name':feature_name,'importance':importance} ).sort_values(by = 'importance', ascending = False)

    feature_importance.to_csv(f'{model_and_feats_file}/{feature_importance_file_name}.csv',index=False)
    
    print('save feature inmportance done !!!!!!!!!!!')

In [31]:
def train_model_with_all_data(train_data: pd.core.frame.DataFrame, # train数据
                              val_data: pd.core.frame.DataFrame, # validation数据 可以传None
                             label: str, # 训练标签 0/1
                             features: list, # 训练特征集
                             model_and_feats_file: str, # 模型和特征重要度文件存储路径
                             model_file_name: str, # 模型名称
                             feature_importance_file_name: str, # 特征重要度文件名称
                             is_alldata_train: bool, # 最后是否选择全量样本训练
                             is_early_stoppping: bool, # 是否使用early-stopping
                             num_boost_round: int, # 最大训练轮数
                             params: dict # 训练参数
                             ):
    
    print(f'feature_name len: {len(features)}')
    
    print('get bst_num_boost_round')
    
    if is_early_stoppping:
    
        lgb_model_train_and_save(train_data, # train数据
                                 val_data, # validation数据 可以传None
                                 label, # 训练标签 0/1
                                 features, # 训练特征集
                                 model_and_feats_file, # 模型和特征重要度文件存储路径
                                 f'{model_file_name}_mid', # 模型名称
                                 f'{feature_importance_file_name}_mid', # 特征重要度文件名称
                                 is_early_stoppping, # 是否使用early-stopping
                                 num_boost_round, # 最大训练轮数
                                 params # 训练参数
                                )

        model_file_name_bst_num = f'{model_file_name}_mid'

        model = lgb.Booster(model_file = f'{model_and_feats_file}/{model_file_name_bst_num}.txt')

        bst_num_boost_round = int(model.num_trees() * 1.1)

        print(f'early_stoppping_true bst_num_boost_round: {bst_num_boost_round}')

    else:
        
        bst_num_boost_round = num_boost_round
        print(f'early_stoppping_false bst_num_boost_round: {num_boost_round}')
        
        lgb_model_train_and_save(train_data, # train数据
                                 val_data, # validation数据 可以传None
                                 label, # 训练标签 0/1
                                 features, # 训练特征集
                                 model_and_feats_file, # 模型和特征重要度文件存储路径
                                 f'{model_file_name}_mid', # 模型名称
                                 f'{feature_importance_file_name}_mid', # 特征重要度文件名称
                                 is_early_stoppping, # 是否使用early-stopping
                                 num_boost_round, # 最大训练轮数
                                 params # 训练参数
                                )
        
    
    if is_alldata_train:
        
        print('train final model with all data !!!')
        
        # is_early_stoppping 置为False
        is_early_stoppping = False
        
        if val_data is not None:
            # 将train_data 和 val_data 拼接为完整样本
            train_data = pd.concat([train_data, val_data])
            val_data = None

        lgb_model_train_and_save(train_data, # 训练数据
                                 val_data, # validation数据 可以传None
                                 label, # 训练标签 0/1
                                 features, # 训练特征集
                                 model_and_feats_file, # 模型和特征重要度文件存储路径
                                 model_file_name, # 模型名称
                                 feature_importance_file_name, # 特征重要度文件
                                 is_early_stoppping, # 是否使用early-stopping
                                 bst_num_boost_round, # 最佳训练轮数
                                 params # 训练参数
                                )
    
    print('finished !!!')
    

# predict code

In [20]:
def get_all_data_train_prediction(model_and_feats_file, #模型和特征重要度文件存储路径 
                              model_file_name, # 模型名称
                              method, # 模型预估结果名称
                              test_data_with_features, # 测试集
                                     save_file_name # 预估结果存储地址
                                 ):
    
    
    model = lgb.Booster(model_file = f'{model_and_feats_file}/{model_file_name}.txt')

    feature_name = model.feature_name()


    print(f'feature name len: {len(feature_name)}')

    test_data_with_features[f'{method}'] = model.predict(test_data_with_features[feature_name], num_threads = 60)

#     predict_list.append(f'{method}_pred')

    
    print('save predict result !!!')
    test_data_with_features[['trace_id', f'{method}']].to_parquet(f'{save_file_name}')
    
    
    print('save predict result done !!!')

# 训练 or 测试 数据预处理（主模型V6 长风险版 训练集和测试集获取 示例）

## 数据集说明

In [ ]:
'''
复贷全场景长风险主模型V6  训练和OOT 划分说明

训练使用标签: current_456overdue7_ever 

训练集放款日: 20240101~20250531所有复贷样本

特征集： 特征包括APP/SMS白名单/首贷征信/鉴权/贷中行为特征/社保/三方多头类特征，触发社保_多头AdvanceV2_同盾多头V1数据采集。

OOT: 20250601

'''

In [ ]:
'''

注释：以下介绍的数据集均存在root路径下，后续如果有其它数据需要存储，可以存储在/data1 路径下（额外挂载的磁盘）

1、数据集1: 用于构建训练和validation集合

数据表成功放款在20230101~20260217区间的复贷成功放款样本（包含特征和业务字段以及主模型V6模型分【注释：模型分从25年6月开始有值】）

地址：/data1/mex_reloan_data/mex_reloan_trace_payout_test_data_s20230101_e20260217_v6features_label_basicinfo （parquet格式，可以根据需要抽取部分字段）


2、主模型V6 训练（train_data） 和 validation（val_data） 数据集

train_data地址： /data1/mex_reloan_data/mex_reloan_main_v6_train_data
val_data地址：/data1/mex_reloan_data/mex_reloan_main_v6_validation_data 


3、数据集2 主模型V6-OOT-s20250601~e20260228 全量（放款 + 未放款）样本特征集

地址：/data1/mex_reloan_data/mex_reloan_trace_test_data_s20250601_e20260228_v6features （parquet格式）


4、数据集3 主模型V6-OOT-s20250601~e20260228 全量（放款 + 未放款）样本 基础信息（业务字段）+ label + 主模型V6长短风险线上模型分

地址：/data1/mex_reloan_data/mex_reloan_trace_test_data_s20250601_e20260228_basicinfo_and_v6modelscore （parquet格式）


5、数据集4  特征描述

特征描述表：包括特征名称（etl_feature_name）和描述（description）

地址：/data1/mex_reloan_data/复贷主模型V6入模特征描述.csv

'''

## 获取特征描述

In [6]:
feature_desc = pd.read_csv('/data1/mex_reloan_data/复贷主模型V6入模特征描述.csv', sep = ',')
feature_desc.shape

(1833, 2)

In [7]:
feature_desc.head(1)

,etl_feature_name,description
0,MEXICO_AUTH___applyLoanAge,申请时的年龄


In [8]:
features = feature_desc['etl_feature_name'].tolist()
len(features)

1833

## 获取训练集

### 主模型V6-全量训练集获取

In [ ]:
'''
使用数据集1 获取主模型V6 训练集

成功放款时间在2024-01-01~2025-05-31之间的样本

训练时，训练集均满足6期表现

短风险模型采用 '1pd7' 作为标签，长风险使用'current_456overdue7_ever' 作为标签

'''

In [10]:
data = pd.read_parquet('/data1/mex_reloan_data/mex_reloan_trace_payout_test_data_s20230101_e20260217_v6features_label_basicinfo')
data.shape

(3175334, 2092)

In [11]:
short_risk_label = '1pd7'  # 短风险训练标签
long_risk_label = 'current_456overdue7_ever'  # 长风险训练标签
split_key = 'current_payout_order_payout_date' # 成功放款日期
split_start_date = '2024-01-01'  # 训练集开始日期
split_end_date = '2025-05-31' # 训练集结束日期
train_samples = data[(data[split_key] >= split_start_date) 
                    & (data[split_key] <= split_end_date) 
                     & (data[short_risk_label] >= 0)
                     & (data[long_risk_label] >= 0)
                    ]
train_samples.shape

(1572813, 2092)

In [12]:
del data
gc.collect()

0

In [13]:
train_samples['payout_month'] = train_samples['current_payout_order_payout_date'].apply(lambda x: x[:7])
train_samples['payout_month'].value_counts()

2024-07    119820
2024-08    117692
2024-06    112629
2024-05    104956
2024-03    104112
2024-09    100036
2024-04     97637
2024-02     95446
2024-01     92720
2025-05     90711
2025-03     89298
2025-04     87531
2024-10     81818
2025-02     75937
2025-01     70110
2024-12     66468
2024-11     65892
Name: payout_month, dtype: int64

In [14]:
# 正样本比例
train_samples.groupby(['payout_month']).agg({'trace_id': 'count',
                                            short_risk_label: 'mean',
                                            long_risk_label: 'mean'})

,trace_id,1pd7,current_456overdue7_ever
payout_month,,,
2024-01,92720,0.058154,0.290293
2024-02,95446,0.054785,0.311129
2024-03,104112,0.061847,0.326629
2024-04,97637,0.064340,0.322859
2024-05,104956,0.063760,0.335865
2024-06,112629,0.063456,0.359961
2024-07,119820,0.068895,0.376465
2024-08,117692,0.075630,0.371809
2024-09,100036,0.060568,0.317806


### 主模型V6-训练集划分train_data 和 val_data

In [15]:
train_data, val_data = train_test_split(train_samples, # 全量训练集
                                        test_size = 0.25, # validation比例
                                        random_state=30, # 随机种子
                                        shuffle=True # 是否shuffle
                                       )
print(f'train_data shape: {train_data.shape}; val_data shape: {val_data.shape}')

train_data shape: (1179609, 2093); val_data shape: (393204, 2093)


In [16]:
# 存储主模型V6-train_data
train_data.to_parquet('/data1/mex_reloan_data/mex_reloan_main_v6_train_data')

In [17]:
# 存储主模型V6-val_data
val_data.to_parquet('/data1/mex_reloan_data/mex_reloan_main_v6_validation_data')

## 使用数据集2 获取OOT全量（放款 + 未放款）样本（s20250601 ~ e20260228）

In [ ]:
'''
注释：如果仅需要下单或者成功放款样本，可以使用is_order 或者 is_payout 进行筛选
'''

In [18]:
test_samples = pd.read_parquet('/data1/mex_reloan_data/mex_reloan_trace_test_data_s20250601_e20260228_v6features')
test_samples.shape

(5273303, 1842)

In [19]:
test_samples.columns

Index(['userId', 'loanAccountId', 'orderId', 'trace_id', 'time_stamp', 'APP_EMBEDDING_V2_V7_FEATURE___features___embeddingAvgV7AppLoanF72', 'MEXICO_SMS_SUB_MODEL___features___smsV1TotalBankSmsOutflowSmsCount', 'APP_EMBEDDING_V2_V7_FEATURE___features___embeddingAvgV7AppLoanF78', 'MEXICO_SMS_SUB_MODEL___features___smsV1TotalBankSmsInflowSmsRatio', 'APP_EMBEDDING_V2_V7_FEATURE___features___embeddingAvgV7AppLoanF81',
       ...
       'MEXICO_MULTI_LOAN_ORDER_INFO___multiLoanOrders___completePrincipalVsLatestRemainCreditRatio', 'MEXICO_MULTI_LOAN_ORDER_INFO___multiLoanOrders___completePrincipal', 'MEXICO_CIRCULO_CREDIT_FIRST_LOAN___creditAccountOpenInfo60d___creditTypeStatRiskVO___personalLoanCoreStat___maxCreditRiskVO___min', 'APP_EMBEDDING_V2_V7_FEATURE___features___embeddingAvgV7AppLoanF54', 'MEXICO_SMS_SUB_MODEL___features___smsV1TotalBankSmsOutflowSmsRatio', 'APP_EMBEDDING_V2_V7_FEATURE___features___embeddingAvgV7AppLoanF60', 'trace_month', 'is_order', 'is_payout', 'risk_type'], dtype

# 模型训练

In [ ]:
'''
注释：

以下提供了2种训练模式

模式一：根据early-stopping 寻找最优训练轮数（树的棵树）

模式二：直接提供固定轮数（树的棵树）训练

模式选择tips:

复贷样本的训练集的中，存在同一个用户多笔坏样本的情况

（1）如果使用短风险标签（例如1pd7），同一用户多笔坏情况少，使用early-stopping可以找到比较合适的训练轮数

（2）如果使用长风险标签（例如4pd7、6pd7），同一用户多笔坏情况多，使用early-stopping，
    
    如果最大训练轮数设置的足够大，会存在训练无法终止的情况，这种情况建议使用固定轮数进行训练，避免过拟合情况


'''

## early-stopping 训练示例 （主模型V6-短风险版 重train）

In [ ]:
'''
模型训练说明：
1、如果训练提供了train_data和val_data, 则直接使用训练，
   否则先将训练集【！！！ 如果只传train_data, val_data 需传入None ！！！】拆分为train 和 val两部分，
   根据val上AUC效果，使用early-stopping确定最佳训练轮数
2、使用全量训练集进行训练，根据1得到的最佳训练轮数，乘1.1（可以自己调节）作为全量训练（如果需要）的训练轮数
'''

In [25]:
# 获取train_data和val_data
train_data = pd.read_parquet('/data1/mex_reloan_data/mex_reloan_main_v6_train_data')
val_data = pd.read_parquet('/data1/mex_reloan_data/mex_reloan_main_v6_validation_data')
print(f'train_data shape: {train_data.shape}; val_data shape: {val_data.shape}')

train_data shape: (1179609, 2093); val_data shape: (393204, 2093)


In [34]:
# 模型训练入参

# 训练标签
label = '1pd7'
print(f'训练label: {label}')

# 计算scale_pos_weight（入模参数，平衡正负样本比例）
scale_pos_weight = round((1 - train_samples[label].mean())/train_samples[label].mean(), 2)
print(f'scale_pos_weight: {scale_pos_weight}')

# 模型训练轮数
num_boost_round = 30000
print(f'模型最大训练轮数: {num_boost_round}')

# 是否使用early_stoppping
is_early_stoppping = True
print(f'是否使用early_stoppping: {is_early_stoppping}')

# 最后是否选择全量样本训练 建议设置为true，当前训练集量级，排除掉validation训练会损失部分效果
# 如果设置为true，仅提供train_data, val_data为None，那train_data就作为全量样本训练
# 如果train_data 和 val_data均提供，就使用二者拼接后的全量样本进行训练
is_alldata_train = True
print(f'是否选择全量样本训练is_alldata_train: {is_alldata_train}')

# 模型训练参数
params = {
            "boosting_type": 'gbdt', # 默认
            "objective": "binary", 
            "metric": "auc", # 评估函数
            "learning_rate": 0.02, # 学习率
            "bagging_fraction": 0.8, # 训练集采样比例
            "bagging_freq": 100, # 训练集采样间隔轮数
            "min_child_weight": 1000, 
            "scale_pos_weight": scale_pos_weight,  # 正负样本均衡参数
            "max_depth": 4, # 树深
            'num_leaves': 15,  # 单棵树的最大叶子树，31默认 
    #         "gamma": 1,
            "reg_alpha": 0.5, # L2正则项
            "reg_lambda": 20, # 0.8 # L1正则项
            "colsample_bytree": 0.8, # 训练特征列采样比例
            "verbose": -1, 
            'min_data_in_bin': 2000, 
            'bin_construct_sample_cnt': 100000, # 默认
    #         "nthread":60
            'num_threads': 80, # cpu-cores
    #         'subsample': 0.8,
            # 'feature_contri':feature_contri,
            "device_type": 'cpu',
#             "gpu_device_id": 6, # 使用GPU时配置
#             'gpu_platform_id': 0, # 使用GPU时配置
            'random_state': 123
        }



# 模型存储信息 

model_and_feats_file = '/data1/mex_reloan_data/主模型V6/models_and_feats'   # 模型文件存储路径
model_file_name = 'reloan_clear_model_v6_shortvintage_alldata_lgb_train' # 全量数据训练的模型名称
feature_importance_file_name = 'reloan_clear_model_v6_shortvintage_alldata_lgb_train_fgain' # 量数据训练的模型输出的特征重要性文件名称

# 中间结果模型，model_file_name 和 feature_importance_file_name 上自动加_mid后缀
# f'{model_file_name}_mid', # 仅train_data（不包含val_data）训练的模型名称
# f'{feature_importance_file_name}_mid', # 仅train_data（不包含val_data）训练的模型的特征重要度文件名称


训练label: 1pd7
scale_pos_weight: 16.89
模型最大训练轮数: 30000
是否使用early_stoppping: True
是否选择全量样本训练is_alldata_train: True


In [35]:
# 模型训练
t1 = time.time()

train_model_with_all_data(train_data, # train数据
                            val_data, # validation数据 可以传None
                             label, # 训练标签 0/1
                             features, # 训练特征集
                             model_and_feats_file, # 模型和特征重要度文件存储路径
                             model_file_name, # 模型名称
                             feature_importance_file_name, # 特征重要度文件名称
                             is_alldata_train, # 最后是否选择全量样本训练
                             is_early_stoppping, # 是否使用early-stopping
                             num_boost_round, # 最大训练轮数
                             params # 训练参数
                         )

t2 = time.time()
print(f'训练耗时：{(t2 - t1)/60} 分钟')

feature_name len: 1833
get bst_num_boost_round
Training until validation scores don't improve for 100 rounds
[600]	training's auc: 0.73876	valid_1's auc: 0.725717
[1200]	training's auc: 0.755129	valid_1's auc: 0.733357
[1800]	training's auc: 0.767419	valid_1's auc: 0.736695
[2400]	training's auc: 0.777955	valid_1's auc: 0.738932
[3000]	training's auc: 0.787568	valid_1's auc: 0.740145
[3600]	training's auc: 0.796487	valid_1's auc: 0.740844
[4200]	training's auc: 0.804961	valid_1's auc: 0.741251
[4800]	training's auc: 0.812527	valid_1's auc: 0.741632
[5400]	training's auc: 0.82005	valid_1's auc: 0.741915
Early stopping, best iteration is:
[5557]	training's auc: 0.821997	valid_1's auc: 0.741981
save model done !!!!!!!!!!!
save feature inmportance done !!!!!!!!!!!
early_stoppping_true bst_num_boost_round: 6112
train final model with all data !!!
[600]	training's auc: 0.736183
[1200]	training's auc: 0.750912
[1800]	training's auc: 0.761401
[2400]	training's auc: 0.770497
[3000]	training's a

## 固定轮数 训练示例 （主模型V6-长风险版 重train）

In [ ]:
'''
注释：主模型V6-长风险版原模型是使用XGB进行训练的，当前大部分模型都是使用LGB训练的，LGB在内存占用上更有优势，这里参考原版，使用LGB进行一个重train

原版采用固定轮数训练，这里也采用固定轮数10000轮进行训练
'''

In [36]:
# 模型训练入参

# 训练标签
label = 'current_456overdue7_ever'
print(f'训练label: {label}')

# 计算scale_pos_weight（入模参数，平衡正负样本比例）
scale_pos_weight = round((1 - train_samples[label].mean())/train_samples[label].mean(), 2)
print(f'scale_pos_weight: {scale_pos_weight}')

# 模型训练轮数-长风险标签采用固定轮数进行训练
num_boost_round = 10000
print(f'模型训练轮数: {num_boost_round}')

# 是否使用early_stoppping
is_early_stoppping = False
print(f'是否使用early_stoppping: {is_early_stoppping}')

# 最后是否选择全量样本训练 建议设置为true，当前训练集量级，排除掉validation训练会损失部分效果
# 如果设置为true，仅提供train_data, val_data为None，那train_data就作为全量样本训练
# 如果train_data 和 val_data均提供，就使用二者拼接后的全量样本进行训练
is_alldata_train = True
print(f'是否选择全量样本训练is_alldata_train: {is_alldata_train}')

# 模型训练参数
params = {
            "boosting_type": 'gbdt', # 默认
            "objective": "binary", 
            "metric": "auc", # 评估函数
            "learning_rate": 0.02, # 学习率
            "bagging_fraction": 0.8, # 训练集采样比例
            "bagging_freq": 100, # 训练集采样间隔轮数
            "min_child_weight": 1000, 
            "scale_pos_weight": scale_pos_weight,  # 正负样本均衡参数
            "max_depth": 4, # 树深
            'num_leaves': 15,  # 单棵树的最大叶子树，31默认 
    #         "gamma": 1,
            "reg_alpha": 0.5, # L2正则项
            "reg_lambda": 20, # 0.8 # L1正则项
            "colsample_bytree": 0.8, # 训练特征列采样比例
            "verbose": -1, 
            'min_data_in_bin': 2000, 
            'bin_construct_sample_cnt': 100000, # 默认
    #         "nthread":60
            'num_threads': 80, # cpu-cores
    #         'subsample': 0.8,
            # 'feature_contri':feature_contri,
            "device_type": 'cpu',
#             "gpu_device_id": 6, # 使用GPU时配置
#             'gpu_platform_id': 0, # 使用GPU时配置
            'random_state': 123
        }



# 模型存储信息
model_and_feats_file = '/data1/mex_reloan_data/主模型V6/models_and_feats'   # 模型文件存储路径
model_file_name = 'reloan_clear_model_v6_longvintage_alldata_lgb_train_fixedr1w' # 模型名称
feature_importance_file_name = 'reloan_clear_model_v6_longvintage_alldata_lgb_train_fixedr1w_fgain' # 模型输出的特征重要性文件名称

# 中间结果模型，model_file_name 和 feature_importance_file_name 上自动加_mid后缀
# f'{model_file_name}_mid', # 仅train_data（不包含val_data）训练的模型名称
# f'{feature_importance_file_name}_mid', # 仅train_data（不包含val_data）训练的模型的特征重要度文件名称


训练label: current_456overdue7_ever
scale_pos_weight: 2.34
模型训练轮数: 10000
是否使用early_stoppping: False
是否选择全量样本训练is_alldata_train: True


In [37]:
# 模型训练
t1 = time.time()

train_model_with_all_data(train_data, # train数据
                            val_data, # validation数据 可以传None
                             label, # 训练标签 0/1
                             features, # 训练特征集
                             model_and_feats_file, # 模型和特征重要度文件存储路径
                             model_file_name, # 模型名称
                             feature_importance_file_name, # 特征重要度文件名称
                             is_alldata_train, # 最后是否选择全量样本训练
                             is_early_stoppping, # 是否使用early-stopping
                             num_boost_round, # 最大训练轮数
                             params # 训练参数
                         )

t2 = time.time()
print(f'训练耗时：{(t2 - t1)/60} 分钟')

feature_name len: 1833
get bst_num_boost_round
early_stoppping_false bst_num_boost_round: 10000
Training until validation scores don't improve for 100 rounds
[600]	training's auc: 0.725037	valid_1's auc: 0.721097
[1200]	training's auc: 0.734756	valid_1's auc: 0.728484
[1800]	training's auc: 0.74118	valid_1's auc: 0.732634
[2400]	training's auc: 0.746464	valid_1's auc: 0.735685
[3000]	training's auc: 0.750994	valid_1's auc: 0.738074
[3600]	training's auc: 0.75499	valid_1's auc: 0.740008
[4200]	training's auc: 0.758862	valid_1's auc: 0.741838
[4800]	training's auc: 0.762648	valid_1's auc: 0.743544
[5400]	training's auc: 0.766004	valid_1's auc: 0.744931
[6000]	training's auc: 0.769298	valid_1's auc: 0.746334
[6600]	training's auc: 0.772642	valid_1's auc: 0.747655
[7200]	training's auc: 0.775848	valid_1's auc: 0.74889
[7800]	training's auc: 0.778817	valid_1's auc: 0.75003
[8400]	training's auc: 0.78186	valid_1's auc: 0.751107
[9000]	training's auc: 0.784801	valid_1's auc: 0.752194
[9600]	t

### gpu训练-test （对比速度，gpu可以快一倍）

In [43]:
# 模型训练入参

# 训练标签
label = 'current_456overdue7_ever'
print(f'训练label: {label}')

# 计算scale_pos_weight（入模参数，平衡正负样本比例）
scale_pos_weight = round((1 - train_samples[label].mean())/train_samples[label].mean(), 2)
print(f'scale_pos_weight: {scale_pos_weight}')

# 模型训练轮数-长风险标签采用固定轮数进行训练
num_boost_round = 10000
print(f'模型训练轮数: {num_boost_round}')

# 是否使用early_stoppping
is_early_stoppping = False
print(f'是否使用early_stoppping: {is_early_stoppping}')

# 最后是否选择全量样本训练 建议设置为true，当前训练集量级，排除掉validation训练会损失部分效果
# 如果设置为true，仅提供train_data, val_data为None，那train_data就作为全量样本训练
# 如果train_data 和 val_data均提供，就使用二者拼接后的全量样本进行训练
is_alldata_train = True
print(f'是否选择全量样本训练is_alldata_train: {is_alldata_train}')

# 模型训练参数
params = {
            "boosting_type": 'gbdt', # 默认
            "objective": "binary", 
            "metric": "auc", # 评估函数
            "learning_rate": 0.02, # 学习率
            "bagging_fraction": 0.8, # 训练集采样比例
            "bagging_freq": 100, # 训练集采样间隔轮数
            "min_child_weight": 1000, 
            "scale_pos_weight": scale_pos_weight,  # 正负样本均衡参数
            "max_depth": 4, # 树深
            'num_leaves': 15,  # 单棵树的最大叶子树，31默认 
    #         "gamma": 1,
            "reg_alpha": 0.5, # L2正则项
            "reg_lambda": 20, # 0.8 # L1正则项
            "colsample_bytree": 0.8, # 训练特征列采样比例
            "verbose": -1, 
            'min_data_in_bin': 2000, 
            'bin_construct_sample_cnt': 100000, # 默认
    #         "nthread":60
            'num_threads': 80, # cpu-cores
    #         'subsample': 0.8,
            # 'feature_contri':feature_contri,
            "device_type": 'gpu',
            "gpu_device_id": 6, # 使用GPU时配置
            'gpu_platform_id': 0, # 使用GPU时配置
            'random_state': 123
        }



# 模型存储信息
model_and_feats_file = '/data1/mex_reloan_data/主模型V6/models_and_feats'   # 模型文件存储路径
model_file_name = 'reloan_clear_model_v6_longvintage_alldata_lgb_gpu_train_fixedr1w' # 模型名称
feature_importance_file_name = 'reloan_clear_model_v6_longvintage_alldata_lgb_gpu_train_fixedr1w_fgain' # 模型输出的特征重要性文件名称

# 中间结果模型，model_file_name 和 feature_importance_file_name 上自动加_mid后缀
# f'{model_file_name}_mid', # 仅train_data（不包含val_data）训练的模型名称
# f'{feature_importance_file_name}_mid', # 仅train_data（不包含val_data）训练的模型的特征重要度文件名称


训练label: current_456overdue7_ever
scale_pos_weight: 2.34
模型训练轮数: 10000
是否使用early_stoppping: False
是否选择全量样本训练is_alldata_train: True


In [44]:
# 模型训练
t1 = time.time()

train_model_with_all_data(train_data, # train数据
                            val_data, # validation数据 可以传None
                             label, # 训练标签 0/1
                             features, # 训练特征集
                             model_and_feats_file, # 模型和特征重要度文件存储路径
                             model_file_name, # 模型名称
                             feature_importance_file_name, # 特征重要度文件名称
                             is_alldata_train, # 最后是否选择全量样本训练
                             is_early_stoppping, # 是否使用early-stopping
                             num_boost_round, # 最大训练轮数
                             params # 训练参数
                         )

t2 = time.time()
print(f'训练耗时：{(t2 - t1)/60} 分钟')

feature_name len: 1833
get bst_num_boost_round
early_stoppping_false bst_num_boost_round: 10000
Training until validation scores don't improve for 100 rounds
[600]	training's auc: 0.725103	valid_1's auc: 0.721145
[1200]	training's auc: 0.734786	valid_1's auc: 0.728484
[1800]	training's auc: 0.741221	valid_1's auc: 0.732691
[2400]	training's auc: 0.746475	valid_1's auc: 0.735754
[3000]	training's auc: 0.751036	valid_1's auc: 0.738162
[3600]	training's auc: 0.755038	valid_1's auc: 0.740106
[4200]	training's auc: 0.758899	valid_1's auc: 0.741893
[4800]	training's auc: 0.762708	valid_1's auc: 0.743615
[5400]	training's auc: 0.766066	valid_1's auc: 0.745006
[6000]	training's auc: 0.769379	valid_1's auc: 0.7464
[6600]	training's auc: 0.772712	valid_1's auc: 0.74774
[7200]	training's auc: 0.775895	valid_1's auc: 0.748955
[7800]	training's auc: 0.778902	valid_1's auc: 0.750055
[8400]	training's auc: 0.781906	valid_1's auc: 0.75115
[9000]	training's auc: 0.784825	valid_1's auc: 0.752248
[9600]	

# 模型预测

## 主模型V6 OOT-s20250601 全量（放款 + 未放款）预估

In [39]:
model_and_feats_file = '/data1/mex_reloan_data/主模型V6/models_and_feats' # 模型文件存储路径
save_path = '/data1/mex_reloan_data/主模型V6/test_prediction' # 预估结果存储路径

for model in [
    'reloan_clear_model_v6_shortvintage_alldata_lgb_train',
    'reloan_clear_model_v6_longvintage_alldata_lgb_train_fixedr1w', # 模型名称
]:
    print(model)
    save_file_name = f'{save_path}/{model}_s2506_e2602_reloan_all_trace_prediction'  # 模型预估结果存储文件名称
    
    # 模型预估&结果存储（仅存储模型分和trace_id）
    get_all_data_train_prediction(model_and_feats_file, #模型和特征重要度文件存储路径 
                          model, # 模型名称
                          model, # 模型预估结果名称
                          test_samples, # 测试集
                                 save_file_name # 预估结果存储地址
                                 )

reloan_clear_model_v6_shortvintage_alldata_lgb_train
feature name len: 1833
save predict result !!!
save predict result done !!!
reloan_clear_model_v6_longvintage_alldata_lgb_train_fixedr1w
feature name len: 1833
save predict result !!!
save predict result done !!!


# 特征重要性

## 测试用例

In [41]:
model_and_feats_file = '/data1/mex_reloan_data/主模型V6/models_and_feats'  # 模型和特征重要度文件存储路径 
model_file_name = 'reloan_clear_model_v6_longvintage_alldata_lgb_train_fixedr1w' # 模型名称
feature_importance_file_name = 'reloan_clear_model_v6_longvintage_alldata_lgb_train_fixedr1w_fgain' # 模型输出的特征重要性文件名称

feature_importance = pd.read_csv(f'{model_and_feats_file}/{feature_importance_file_name}.csv') # 特征重要性倒序排列

# 特征重要性占比
feature_importance['importance_rate'] = feature_importance['importance']/feature_importance['importance'].sum()

# 关联特征描述
feature_desc = feature_desc.rename(columns = {'etl_feature_name': 'feature_name'})
feature_importance = pd.merge(feature_importance, feature_desc, how = 'inner', on = 'feature_name')

print(f'feature_importance : shape: {feature_importance.shape}')

feature_importance : shape: (1833, 4)


In [42]:
feature_importance.head(100)

,feature_name,importance,importance_rate,description
0,MEXICO_ADVANCE_MULTI_PLATFORM_DETECTION_V2___hourFromLastApply,1.428838e+06,0.117684,申请时间和末次调用时间间隔的小时时差
1,MEXICO_ORDER_INFO___recent1SingleOrder___creditUsageRatio,1.133761e+06,0.093381,最近第1笔订单特征_订单额度使用率(实际借款金额/可用额度)
2,MEXICO_CREDITS_CALC_ORDER___latest32D___orderCreditUsageRatio___min,4.082915e+05,0.033628,32天窗口_额度使用率_最小值
3,MEXICO_MULTI_LOAN_ORDER_INFO___multiLoanOrders___incompletePrincipal,2.356639e+05,0.019410,在贷续借订单特征_未结清金额
4,MEXICO_ADVANCE_MULTI_PLATFORM_DETECTION_V2___medianHourFromAllApply,1.582612e+05,0.013035,申请时间和所有调用时间间隔的小时时差中位数
5,MEXICO_ORDER_INFO___stat10000D___completedInstalPrincipalSum,1.218718e+05,0.010038,近10000天的贷中行为特征_历史已结清账单本金总和
6,MEXICO_MULTI_LOAN_ORDER_INFO___multiLoanOrders___creditUsageRatioMax,1.172366e+05,0.009656,在贷续借订单特征_额度使用率最大值
7,MEXICO_MULTI_LOAN_ORDER_INFO___multiLoanOrders___completeVsIncompletePrincipalRatio,1.088189e+05,0.008963,在贷续借订单特征_已结清账单借款金额/未结清账单借款金额）
8,MEXICO_CREDITS_CALC_ORDER___latest384D___latestCompletedPrincipal___avg,1.087339e+05,0.008956,384天窗口_下单时刻的最近一笔订单已还本金_平均值
9,MEXICO_ADVANCE_MULTI_PLATFORM_DETECTION_V2___hourFromLastCallWithPhoneNumber,1.057884e+05,0.008713,申请时间和末次调用时间间隔的小时时差by phoneNumber
